# Parameter Sweeping
Optimize momentum lookback periods and EMA pullback levels.

In [ ]:
import sys
sys.path.append('..')
import vectorbt as vbt
import pandas as pd
import pandas_ta as ta
import numpy as np
from src.portfolio_config import create_portfolio
from strategies.dual_momentum_swing import load_data

In [ ]:
# Load Data
data = load_data('../data')
close = data['close']
high = data['high']
low = data['low']
rsi14 = data['rsi14']
macd_line = data['macd_line']
macd_signal = data['macd_signal']
sma50 = data['sma50']

In [ ]:
# Define Parameter Grids
roc_windows = [60, 90, 120, 180]
ema_windows = [10, 20, 30]

# For simplicity, we will simulate a loop over parameters to find the best Sharpe/Return
results = []
assets = ['nifty', 'gold', 'liquid']

for roc_w in roc_windows:
    for ema_w in ema_windows:
        # Calculate param-specific indicators
        roc = close.apply(lambda x: ta.roc(x, length=roc_w))
        ema = close.apply(lambda x: ta.ema(x, length=ema_w))
        
        # Macro Regime
        max_roc_asset = roc.idxmax(axis=1)
        active_asset = pd.DataFrame(index=roc.index, columns=assets, data=False)
        for asset in assets:
            active_asset[asset] = (max_roc_asset == asset)
            
        # Swing Entry
        pullback = (low <= ema) & (high >= ema)
        proximity = (close <= ema * 1.015) & (close >= ema * 0.985)
        pullback_condition = pullback | proximity
        rsi_condition = (rsi14 > 40) & (rsi14 < 70)
        
        entries = pd.DataFrame(index=close.index, columns=assets, data=False)
        for asset in ["nifty", "gold"]:
            entries[asset] = active_asset[asset] & pullback_condition[asset] & rsi_condition[asset]
        entries["liquid"] = active_asset["liquid"] & (~active_asset["liquid"].shift(1).fillna(False))
        
        # Exits
        macd_cross_below = (macd_line < macd_signal) & (macd_line.shift(1) >= macd_signal.shift(1))
        close_below_sma = close < sma50
        regime_shift = ~active_asset
        
        exits = pd.DataFrame(index=close.index, columns=assets, data=False)
        for asset in ["nifty", "gold"]:
            exits[asset] = macd_cross_below[asset] | close_below_sma[asset] | regime_shift[asset]
        exits["liquid"] = regime_shift["liquid"]
        
        # Simulate
        pf = create_portfolio(close, entries, exits)
        stats = pf.stats()
        
        results.append({
            'ROC_Window': roc_w,
            'EMA_Window': ema_w,
            'Total_Return': stats.get('Total Return [%]', np.nan),
            'Sharpe_Ratio': stats.get('Sharpe Ratio', np.nan),
            'Max_Drawdown': stats.get('Max Drawdown [%]', np.nan)
        })

results_df = pd.DataFrame(results)
display(results_df.sort_values(by='Sharpe_Ratio', ascending=False))